In [38]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

**<font size="6" color="red">ch2_Ollama_LLM활용의 기본개념(LangChain)</font>**

# 1. LLM을 활용하여 답변 생성
## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT(open ai API), Claude같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ollama.com 설치 -> 모델 pull
- cmd창에서 ollama pull deepseek-r1:1.5b

In [10]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')  # cmd에서 pull 하지 않으면 error
result=llm.invoke('What is the capital of France?')
result #AIMessage
# content : 실제 답변
# response_metadata : 모델 실행에 대한 상세 정보(전체소요시간, 모델로딩시간, 처리 토큰수 )

AIMessage(content='\n\nThe capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-09-10T07:59:25.0456678Z', 'done': True, 'done_reason': 'stop', 'total_duration': 365901400, 'load_duration': 3758000, 'prompt_eval_count': 10, 'prompt_eval_duration': 46700000, 'eval_count': 12, 'eval_duration': 305864000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a08a54-2207-71c2-b514-b9f8ffb9c59c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22})

In [9]:
result.response_metadata

{'model': 'deepseek-r1:1.5b',
 'created_at': '2026-09-10T07:56:38.879424Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 15391306000,
 'load_duration': 2181900,
 'prompt_eval_count': 10,
 'prompt_eval_duration': 33493000,
 'eval_count': 531,
 'eval_duration': 15350473000,
 'logprobs': None,
 'model_name': 'deepseek-r1:1.5b',
 'model_provider': 'ollama'}

In [13]:
print(result.usage_metadata)
print(result.content)

{'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22}


The capital of France is Paris.


### 모델 pull
- ollama run llama3.2:1b
- ollama 모델은 공식적으로 한글지원 안됨(llama3.1:405b 한글지원 가능 -> llama3.2:3b 한글 지원이 일부)
- exaone 모델은 공식적으로 한글지원 ollama pull exaone3.5:2.4b

- 모델 저장 경로 : C:\Users\mbc\.ollama


In [5]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="exaone3.5:2.4b")
result = llm.invoke('what is the capital of Republic of Korea?',
#                     temperature=0.2,
#                     top_k=40,
#                     top_p=0.9,
                    #num_ctx=4096 # 컨텍스트 윈도우(입력토큰과 출력토큰)
                   )
print(result.content)

The capital of the Republic of Korea (South Korea) is Seoul.


In [6]:
result = llm.invoke('한국의 수도는 어디야?')
print(result.content)

한국의 수도는 **서울**입니다.


## 2) openai 모델 활용
- pip install langchain-openai


In [11]:
#환경변수 가져오기
import os
from dotenv import load_dotenv
load_dotenv()
print(os.getenv('OPENAI_API_KEY')[:2])

sk


In [14]:
from langchain_openai import ChatOpenAI
llm =ChatOpenAI(model="gpt-4.1-nano")
result=llm.invoke("What is the capital of Korea?")
result = llm.invoke('한국의 수도는 어디야?')
result.content

'한국의 수도는 서울입니다.'

In [15]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")
# result=llm.invoke("What is the capital of Korea?")

TypeError: Anthropic authentication failed: no API key or authorization credentials were provided. Set the ANTHROPIC_API_KEY environment variable, pass api_key=... to ChatAnthropic, or provide credentials via default_headers={"Authorization": ...}. If you are routing through the LangSmith gateway, set LANGSMITH_GATEWAY and LANGSMITH_GATEWAY_API_KEY.

# 2. 렝체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질물
## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate을 사용하여 변수가 포함된 템플릿 작성하면 PromptValue를 만들 수 있다

In [20]:
from langchain_ollama import ChatOllama
llm =ChatOllama(model="llama3.2:1b")
llm.invoke("What is the capital of Korea?").content
llm.invoke("What is the capital of Korea?")
#llm.invoke(0) int타입은 에러발생
#프롬프트 가능 타입: str, PromptValue, or list of BaseMessages

'The capital of Korea is Seoul.'

In [26]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
                                template="what is the capital of {country}?", #{}안에 값을 새로운 값으로 대체
                                input_variables = ['country']
                                )
prompt = prompt_template.invoke({"country":"Korea"})
print(1, prompt)
prompt = prompt_template.invoke('Korea')
print(2, prompt)
llm.invoke(prompt).content

1 text='what is the capital of Korea?'
2 text='what is the capital of Korea?'


'The capital of Korea is Seoul.'

In [28]:
country =input('수도를 알고 싶은 나라는(영어)?')
llm.invoke(prompt_template.invoke(country)).content

수도를 알고 싶은 나라는(영어)?somalia


'The capital of Somalia is Mogadishu.'

In [33]:
def answer(country):
    '나라명을 입력받아 llm에게 수도명을 받아 return'    
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm =ChatOllama(model="llama3.2:1b")
    prompt_template = PromptTemplate(
                                     template='What is the capital of {country}?',
                                     input_variables =['country']
                                    )
    result =llm.invoke(prompt_template.invoke(country))
    return result.content


In [35]:
country = input("수도를 알고 싶은 나라는(영어)?")
print(answer(country))

수도를 알고 싶은 나라는(영어)?iceland
The capital of Iceland is Reykjavik.


## 2) 메세지 기반 프롬프트 작성
- list of BaseMessages
- BaseMessage 상속받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage
- [BaseMessage객체,BaseMessage객체,BaseMessage객체, ...]

In [36]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama
llm= ChatOllama(model="llama3.2:1b")
message_list=[
    SystemMessage(content="You are a helpful assitant!"), #llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Rome"),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Paris"),
    HumanMessage(content="What is the capital of Korea?") #llm에게 질문하고 싶은 진짜 내용
    
]
llm.invoke(message_list).content

'The capital of South Korea is Seoul'

## 3) ChatPromptTemplate 사용(추천; 확장성 용이)
- BaseMessage 리스트 -> 튜플리스트

In [38]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    SystemMessage(content="You are a helpful assitant!"), #llm 페르소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Rome"),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content="The capital of Italy is Paris"),
    HumanMessage(content="What is the capital of {country}?") #llm에게 질문하고 싶은 진짜 내용
    
])
prompt= chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트:', prompt)

프롬프트: messages=[SystemMessage(content='You are a helpful assitant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of {country}?', additional_kwargs={}, response_metadata={})]


In [39]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    #SystemMessage(content="You are a helpful assitant!")
    ("system","You are a helpful assitant!" ), #llm 페르소나
    #HumanMessage(content="What is the capital of Italy?")
    ("human","What is the capital of Italy" ), # 질문 답변 예제(few shot)
    #AIMessage(content="The capital of Italy is Rome")
    ("ai","The capital of Italy is Rome" ),
    #HumanMessage(content="What is the capital of France?")
    ("human","What is the capital of France" ), # 질문 답변 예제(few shot)
    #AIMessage(content="The capital of Italy is Paris"),
    ("ai","The capital of Italy is Paris" ),
    #HumanMessage(content="What is the capital of {country}?") 
    ("human","What is the capital of {country}?")#llm에게 질문하고 싶은 진짜 내용
])
prompt= chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트:', prompt)

프롬프트: messages=[SystemMessage(content='You are a helpful assitant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of France', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Paris', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={})]


In [40]:
llm.invoke(prompt).content

'Korea is actually divided into two separate countries: South Korea and North Korea. The capital of South Korea is Seoul, while the capital of North Korea is Pyongyang.\n\nHowever, if you\'re asking about the administrative capital of Korea, the country has several cities that serve as important centers of government, economy, and culture. Some of the most notable ones include:\n\n* Pyongyang (North Korea): The capital of North Korea, where the government and many government institutions are located.\n* Kaesong (North Korea): A former capital city of North Korea that has been restored to its former glory.\n* Anyang (South Korea): A city in the Gyeonggi Province of South Korea that is often referred to as the "Silicon Valley of South Korea" due to its thriving tech industry.\n\nSo, while there isn\'t a single "capital" of Korea, Seoul is generally considered the most important city in the country.'

In [41]:
llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})).content

'The capital of South Korea is Seoul'

In [42]:
llm.invoke(chatPromptTemplate.invoke({'Korea'})).content  # 변수가 1개일때는 country 생략가능

'The capital of South Korea is Seoul'

# 3. 답변 형식 컨트롤하기
- invoke 실행 결과 AIMessage() 객체로 출력 (but,  내가원하는 출력은 문자일수도 있고 json형식일수도 있고...)
     -> string, json 변환해주는 OutputParser 이용

## 1) 문자열 출력 Parser 이용
- StrOutputParser를 이용하여 LLM출력(AIMessage)를 단순 문자열로 변환

 

In [53]:
# 명시적인 지시사항이 포함된 프롬프트 
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return the name of the city only.",
    input_variables =['country']
)
# 프롬프트 템플릿에 값 주입
prompt =prompt_template.invoke({'country':'Korea'})
print('프롬프트:', prompt)
#llm에 질문
aimessage = llm.invoke(prompt)
#aimessage중 답변만 문자로 받기
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print("Parser결과:",result)

프롬프트: text='What is the capital of Korea? Return the name of the city only.'
Parser결과: Seoul


In [54]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'Korea'})))

'Seoul'

In [55]:
chatPromptTemplate = ChatPromptTemplate([
    ("system","You are a helpful assitant!" ),
    ("human","What is the capital of Italy" ), 
    ("ai","The capital of Italy is Rome" ),
    ("human","What is the capital of France" ), 
    ("ai","The capital of Italy is Paris" ),
    ("human","What is the capital of {country}?Return the name of the city only.")
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'Korea'})))

'Seoul'

## 2)Json 출력 Parser 이용
- {'name':'홍', 'age':20}

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
llm= ChatOllama(model="llama3.2:1b")
country_detail_prompt = PromptTemplate(
                    template="""Give following information about {country},
                    - Capital
                    - Population
                    - Language
                    - Currency
                Return ONLY a valid JSON object with no additional text.
                Example format:
                {{'Capital':'Seoul', 'Population':"50 million", 'Language':'Korean','Currency':'won'}}""",
                    input_variables =['country']
                    )
prompt = country_detail_prompt.invoke({'country':'France'})
aimessage=llm.invoke(prompt)
output_parser = JsonOutputParser()
result = output_parser.invoke(aimessage)
print(type(result),result)

<class 'dict'> {'Capital': 'Paris', 'Population': '67,281,506', 'Language': 'French', 'Currency': 'Euro'}


In [ ]:
country_detail_prompt

In [8]:
output_parser = JsonOutputParser()
output_parser.invoke(llm.invoke(country_detail_prompt.invoke('France')))

{'Capital': 'Paris',
 'Population': '67 million',
 'Language': 'French',
 'Currency': 'Euro'}

## 3) 구조화된 객체로 반환 
- Pydantic 모델 (pip install pydantic)을 사용하여 LLM출력을 구조화된 형식으로 받기(JsonParser보다 더 안정적)
- Pydantic : 데이터 유효성 검사 , 설정관리를 간편하게 해주는 라이브러리


In [11]:
class User:
    def __init__(self, id, name, is_active=True):
        self.id =id
        self.name = name
        self.is_active = is_active

user = User('1','홍길동')
user = User(1,'홍')
print(user)
print(user.id, user.name, user.is_active)

1 홍 True


In [16]:
from pydantic import BaseModel, Field
class User(BaseModel):
    #gt=0 : id>0, lt = 0 : id<0, ge=0 :id>=0, le=0: id<=0
    id:int            = Field(gt=0        , description ='id')
    name:str          = Field(min_length=2, description ='name')
    is_active:bool    = Field(default=True, description= 'id활성화')
user =User(id="1", name='홍길동', is_active=True)
print(user)

id=1 name='홍길동' is_active=True


In [23]:
country_detail_prompt = PromptTemplate(
                    template="""Give following information about {country},
                    - Capital
                    - Population
                    - Language
                    - Currency
                Return ONLY a valid JSON object with no additional text.
                """,
                    input_variables =['country']
                    )
class CountryDetail(BaseModel):
    capital:str =Field(description= "the capital of the country")
    population:int=Field(description="the population of the country")
    language:str=Field(description= "the language of the country")
    currency:str=Field(description= "the currency of the country")
# 출력Parser + llm
structedllm = llm.with_structured_output(CountryDetail)
info = structedllm.invoke(country_detail_prompt.invoke({'Korea'}))
with_structured_output
print(info.capital, info.population, info.language, info.currency)

Seoul 51000000 Korean Korean won


In [24]:
print(info.model_dump()) # 객체를 dict로

{'capital': 'Seoul', 'population': 51000000, 'language': 'Korean', 'currency': 'Korean won'}


# 4. LCEL(LangChain Expression Language)을 활용한 렝체인 생성
## 1) 문자열 출력 파서  사용
- StrOutputParser, ChatOllama, PromptTemplate등은 모두 Runable로 상속받아 invoke가 있음

In [25]:
# 명시적인 지시사항이 포함된 프롬프트 
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(
    template="What is the capital of {country}? Return the name of the city only.",
    input_variables =['country']
)
outputParser=StrOutputParser()
outputParser.invoke(llm.invoke(prompt_template.invoke({'country':'Ghana'})))

'Accra'

## 2) LCEL을 사용한 체인 구성

In [26]:
# 프롬프트 템플릿 -> llm -> 출력파서를 연결시키는 체인 생성
capital_chain = prompt_template | llm | outputParserr
#생성된 체인 invoke
capital_chain.invoke({'country':'Korea'})

'Seoul'

## 3) 복합 체인 구성
- 여러 단계의 추론이 필요한 경우(체인 연결)


In [28]:
#나라 설명 -> 나라이름
country_prompt =PromptTemplate(
                        template="""
                        Guess the name of the country based on th following information:
                        {information}
                        Return the name of the country only
                        """,
                        input_variables=['information']
)
outputParser.invoke(llm.invoke(country_prompt.invoke({'information':"This country is very famous for its wine"})))

'France'

In [29]:
#나라명 추측체인
country_chain= country_prompt | llm | outputParser
country_chain.invoke({'information':"This country is very famous for its wine"})

'Italy'

In [30]:
#국가설명 -> 국가명 -> 그 국가 수도명
final_chain = country_chain | capital_chain
final_chain.invoke({'information':"This country is very famous for its wine"})


'Rome'

In [31]:
final_chain = {'country':country_chain} | capital_chain
final_chain.invoke({'information':"This country is very famous for its wine"})

'Rome'

In [37]:
final_chain.invoke({'information': '이나라는 축구가 유명한 나라 입니다'})

'Tokyo'

```
LLM호출 : ChatOllama(llama3.2:1b / exaone3.5:2.4b), ChatOpenAI
LLM호출에 필요한 프롬프트 템플릿 : PromptTemplate, ChatPromptTemplate(few show, 페르소나 설정)
LLM 결과를 변환 outputParser : str / JSON / pydentic 클래스 생성하여 LLM의 Structed outputParser 이용
위 모두 runable로 부터 상속받은 invoke 가능 => LangChain으로 연결 가능(RAG 검색증강 생성)
```
# 5. 생성형 AI 평가 : 
- 첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
- 두번째 체인 :  음식 -> 음식의 레시피
- 최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피